# Evaluation Script: Observed HTTP Traffic

This script produces results for Section 5.8 Loaded Content and Table 5

### Connect to the MongoDB

In [ ]:
import os
import tqdm
#import magic
import urllib
import pymongo
import requests


import pandas as pd

from requests.exceptions import RequestException
from concurrent.futures import ThreadPoolExecutor, as_completed
from urllib.parse import urlparse
from tqdm.auto import tqdm

client = pymongo.MongoClient("mongodb://localhost:27017/")
db = client["webview"]
network_logs_collection = db["network_logs"]
dynamic_api_calls_collection = db["dynamic_api_calls"]

NET_REQUESTS = False

def print_latex_macro(name: str, value: str):
    print(f"\\newcommand{{\\{name}}}{{{value}}}")

### Retrieve the apps using HTTP network traffic

Results for "Loaded Content"

In [5]:
results = network_logs_collection.find({})
amount_apps_using_webview = len(dynamic_api_calls_collection.distinct("source_package_name"))
amount_apps_with_network_logs = results.distinct("package_name")
print_latex_macro("amountAppsObservedHTTPTraffic", f"{len(amount_apps_with_network_logs):,}")
print_latex_macro("amountAppsObservedHTTPTrafficPercent", f"{(len(amount_apps_with_network_logs)/amount_apps_using_webview)*100:.2f}")

\newcommand{\amountAppsObservedHTTPTraffic}{2,790}
\newcommand{\amountAppsObservedHTTPTrafficPercent}{11.08}


### Retrieve the of AIA .crt requests

Results for "Resource Types (1/)"

In [6]:
# get the URLs
cert_urls = set()
cert_urls_package_names = set()
logs = set()
net_logs = network_logs_collection.find({})
http_urls = set()
for log in net_logs:
    package_name = log.get("package_name", "")
    url = log.get("url", "")
    initiator = log.get("initiator", {})
    request_type = log.get("request_type", "")
    parsed_url = urlparse(url)
    # get the hostname
    hostname = parsed_url.hostname
    if url.endswith(".crt") and initiator == "not an origin":
        cert_urls.add(url)
        cert_urls_package_names.add(package_name)
        continue


    logs.add((package_name, url, hostname, initiator, request_type))

print_latex_macro("appsCert", f"{len(cert_urls_package_names):,}")


df_logs = pd.DataFrame(list(logs), columns=["package_name", "url", "hostname", "initiator", "request_type"])

\newcommand{\appsCert}{80}


### Retrieve stats on upgradable URLs

In [10]:
# Load all network_logs into a DataFrame
records = []
for log in tqdm.tqdm(network_logs_collection.find(), desc="Loading network_logs"):
    records.append({
        "package_name": log.get("package_name"),
        "url": log.get("url"),
        "initiator": log.get("initiator"),
        "request_type": log.get("request_type"),
    })

df = pd.DataFrame(records)

initiator_counts = df["initiator"].value_counts(dropna=False)

def initiator_scheme(initiator):
    if not initiator or not isinstance(initiator, str):
        return "unknown"
    if initiator == "not an origin":
        return "not_an_origin"
    if initiator == "null":
        return "null"
    parsed = urlparse(initiator)
    if parsed.scheme in ("http", "https", "file"):
        return parsed.scheme
    return "other"

df["initiator_scheme"] = df["initiator"].apply(initiator_scheme)

Loading network_logs: 81263it [00:00, 423830.23it/s]


##### Assign each resource an initial resource type based on the extension

In [11]:

# Classify URLs into finer-grained resource types by extension.
EXT_MAP = {
    "Script":     {".js", ".mjs"},
    "CSS":    {".css"},
    "Image":  {".png", ".jpg", ".jpeg", ".gif", ".svg", ".webp", ".ico", ".bmp", ".tiff"},
    "Font":   {".woff", ".woff2", ".ttf", ".otf", ".eot"},
    "Video":  {".mp4", ".webm", ".mov", ".m4v", ".avi", ".mkv"},
    "Audio":  {".mp3", ".wav", ".ogg", ".m4a", ".aac", ".flac"},
    "HTML":   {".html", ".htm"},
    "JSON":   {".json"},
    "XML":    {".xml"},
    "PDF":    {".pdf"},
    "Text":   {".txt"},
    "Certificate": {".crt"},
}
EXT_LOOKUP = {ext: kind for kind, exts in EXT_MAP.items() for ext in exts}

# Rows whose HTTPS-initiator cell represents a mixed-content policy weakness
# (sub-resources + framed content; excludes main-frame navigation and the
# heterogeneous "other" bucket).
MIXED_CONTENT_ROWS = {
    "Image", "Script", "CSS", "Font", "Audio", "Video",
    "JSON", "XML", "Text", "PDF", "Certificate", "Subframe",
}

# Explicit row order for the LaTeX tables. Edit this list to reorder rows.
# Any rows present in the data but missing here are appended at the end
# (sorted by request count, descending) with a warning.
ROW_ORDER = [
    "Top-Level Page",
    "Subframe",
    "Image",
    "Script",
    "CSS",
    "HTML",
    "Font",
    "JSON",
    "Audio",
    "Video",
    "Certificate",
    "Text",
    "XML",
    "PDF",
    "other",
]

def url_ext(url):
    if not isinstance(url, str):
        return ""
    path = urlparse(url).path
    return os.path.splitext(path)[1].lower()

def classify_resource(row):
    rt = row["request_type"]
    if rt == "main frame":
        return "Top-Level Page"
    if rt == "subframe":
        return "Subframe"
    ext = url_ext(row["url"])
    return EXT_LOOKUP.get(ext, "other")

df["resource_type"] = df.apply(classify_resource, axis=1)

##### For those without an extension, query the resource to get the resource type

In [17]:
def unknown(url):
    return "unknown"


def sniff_mime_type(url):
    """
    Fetches the first 2048 bytes of a URL and uses magic bytes to sniff the MIME type.
    """
    if pd.isna(url) or not isinstance(url, str):
        return "Invalid URL"
        
    try:
        # Use stream=True so we don't download large files entirely
        with requests.get(url, stream=True, timeout=5) as response:
            # Check if the request was successful before sniffing
            response.raise_for_status()
            
            # Read the first chunk (2048 bytes is usually enough for magic numbers)
            chunk = next(response.iter_content(chunk_size=2048), b"")
            
            if not chunk:
                return "Empty Content"
            
            # Sniff the buffer using python-magic
            mime_type = magic.from_buffer(chunk, mime=True)
            
            if mime_type == 'text/plain':
                # Decode the binary chunk to a string, ignoring weird characters
                text_chunk = chunk.decode('utf-8', errors='ignore').strip()
                
                # Check if it looks like a JSON object {...} or JSON array [...]
                # We check for '":' to ensure it's not just a random text file starting with a bracket
                if (text_chunk.startswith('{') and '":' in text_chunk) or text_chunk.startswith('['):
                    return 'application/json'
            
            return mime_type
            
    except RequestException as e:
        # Catch connection errors, timeouts, and 404s
        return "Network/HTTP Error"
    except Exception as e:
        # Catch any other unexpected errors
        return "Sniffing Error"
    


tqdm.pandas(desc="Sniffing URLs")
mask = df['resource_type'] == 'other'

if NET_REQUESTS == True:
    df.loc[mask, 'sniffed_resource_type'] = df.loc[mask, 'url'].progress_apply(sniff_mime_type)
else:
    df.loc[mask, 'sniffed_resource_type'] = df.loc[mask, 'url'].progress_apply(unknown)
    
sniffed = df['sniffed_resource_type'].fillna('')

df['final_resource_type'] = df['resource_type']

mask = df['resource_type'] == 'other'


df.loc[mask & sniffed.str.startswith('image'), 'final_resource_type'] = 'Image'
df.loc[mask & (sniffed == 'application/json'), 'final_resource_type'] = 'JSON'
df.loc[mask & (sniffed == 'application/javascript'), 'final_resource_type'] = 'Script'
df.loc[mask & (sniffed == 'text/html'), 'final_resource_type'] = 'HTML'
df.loc[mask & (sniffed == 'text/xml'), 'final_resource_type'] = 'XML'
df.loc[mask & (sniffed == 'text/plain'), 'final_resource_type'] = 'Text'
df.loc[mask & sniffed.str.startswith('audio'), 'final_resource_type'] = 'Audio'
df.loc[mask & (sniffed == 'text/javascript'), 'final_resource_type'] = 'Script'

Sniffing URLs: 100%|██████████| 7030/7030 [00:00<00:00, 1827453.18it/s]


##### Save the results as a CSV

In [ ]:
df.to_csv("results-out/resource_breakdown.csv", sep=';')

##### Create the resource distribution table

In [18]:
# Build resource-type x initiator-scheme app-count table and emit LaTeX.
#
# Output format follows the requested template: a single column per scheme
# (count + row-percentage), marker icons next to bold cells via \markWeak
# (MC policy weakening) and \markGap (enforcement gap vs. Chrome), grouped
# rows separated by \midrule/\hline. Counts are unique app counts; row
# and column totals come directly from the data, not from cell sums.
#
# LaTeX preamble must define (also printed below for convenience):
#   \usepackage{xcolor}                       % for \definecolor
#   \usepackage{fontawesome5}                 % for \faIcon
#   \usepackage{array}                        % for \newcolumntype
#   \usepackage{booktabs}                     % for \toprule etc.
#   \definecolor{PolicyRed}{HTML}{A93226}
#   \definecolor{GapBlue}{HTML}{34495E}
#   \newcommand{\markWeak}{\textcolor{PolicyRed}{\faIcon{arrow-circle-down}}}
#   \newcommand{\markGap}{\textcolor{GapBlue}{\faIcon{minus-circle}}}
#   \newcolumntype{R}[1]{>{\raggedleft\arraybackslash}p{#1}}


TOTAL_APPS = 34200

SCHEME_ORDER = ["https", "http", "file", "not_an_origin", "null"]
SCHEME_LABELS = {
    "https":         "HTTPS",
    "http":          "HTTP",
    "file":          "File",
    "not_an_origin": "Browser-initiated",
    "null":          "``null''",
}

# Marker key -> LaTeX command emitted next to the cell.
MARKER_DEFS = {
    "weak": r"\markWeak",   # systematic MC policy weakening
    "gap":  r"\markGap",    # enforcement gap compared to Chrome
}

# Cells to mark: (resource_type, initiator_scheme) -> marker key.
# Marked cells are also bolded.
CELL_MARKERS = {
    ("Top-Level Page", "https"):         "gap",
    ("Top-Level Page", "not_an_origin"): "gap",
    ("Script",   "https"): "weak",
    ("Script",   "file"):  "gap",
    ("Script",   "null"):  "gap",
    ("Subframe", "https"): "weak",
    ("Subframe", "file"):  "gap",
    ("Subframe", "null"):  "gap",
    ("Font",     "https"): "weak",
    ("Font",     "file"):  "gap",
    ("Font",     "null"):  "gap",
    ("CSS",      "https"): "weak",
    ("CSS",      "file"):  "gap",
    ("CSS",      "null"):  "gap",
    ("Image",    "https"): "weak",
    ("Image",    "file"):  "gap",
    ("Image",    "null"):  "gap",
    ("Audio",    "https"): "weak",
    ("Audio",    "file"):  "gap",
    ("Audio",    "null"):  "gap",
    ("Video",    "https"): "weak",
    ("Video",    "null"):  "gap",
}

# Row layout. Entries are processed in order:
#   - "<midrule>" or "<hline>": insert a horizontal rule
#   - str:  resource type name; renders as a data row with that label
#   - dict with keys:
#       "label"       (required): row label
#       "header_only" (bool):     empty cells; just a section header
#       "type"        (str):      single resource type to source data from
#       "combine"     (list[str]): aggregate multiple types into one row
#                                  (unique-app dedup across the listed types)
ROW_LAYOUT = [
    "Top-Level Page",
    "<hline>",
    "Script", "Subframe", "Font", "CSS",
    "<midrule>",
    "Image", "Audio", "Video",
    "<midrule>",
    "Certificate",
    "<midrule>",
    {"label": "Others*",
     "combine": ["HTML", "JSON", "Text", "XML", "PDF"]},
    {"label": "Unknown", "type": "other"},
]

# ------------------------------------------------------------------------
# Data aggregation.

apps_by_type_scheme = (
    df.groupby(["final_resource_type", "initiator_scheme"])["package_name"]
      .nunique()
      .unstack(fill_value=0)
      .reindex(columns=SCHEME_ORDER, fill_value=0)
)
apps_by_type   = df.groupby("final_resource_type")["package_name"].nunique()
apps_by_scheme = df.groupby("initiator_scheme")["package_name"].nunique()

def cell_count(spec_type_or_dict, scheme):
    """spec is either a resource-type string or a layout dict."""
    if isinstance(spec_type_or_dict, str):
        rt = spec_type_or_dict
        return int(apps_by_type_scheme.at[rt, scheme]) if rt in apps_by_type_scheme.index else 0
    if "type" in spec_type_or_dict:
        rt = spec_type_or_dict["type"]
        return int(apps_by_type_scheme.at[rt, scheme]) if rt in apps_by_type_scheme.index else 0
    if "combine" in spec_type_or_dict:
        # Unique apps loading any of the listed types via this scheme
        # (not a cell sum -- so apps using several of the combined types
        # are counted once).
        mask = (
            df["resource_type"].isin(spec_type_or_dict["combine"])
            & (df["initiator_scheme"] == scheme)
        )
        return int(df.loc[mask, "package_name"].nunique())
    return 0

def row_total_count(spec_type_or_dict):
    if isinstance(spec_type_or_dict, str):
        rt = spec_type_or_dict
        return int(apps_by_type.get(rt, 0))
    if "type" in spec_type_or_dict:
        return int(apps_by_type.get(spec_type_or_dict["type"], 0))
    if "combine" in spec_type_or_dict:
        mask = df["resource_type"].isin(spec_type_or_dict["combine"])
        return int(df.loc[mask, "package_name"].nunique())
    return 0

def lookup_marker(spec, scheme):
    """Markers are keyed by resource-type, so only single-type rows get them."""
    if isinstance(spec, str):
        return CELL_MARKERS.get((spec, scheme))
    if "type" in spec:
        return CELL_MARKERS.get((spec["type"], scheme))
    return None

def fmt_cell(n, marker_key=None):
    if n == 0:
        return "--"
    pct = n / TOTAL_APPS * 100
    pct_str = "<0.1" if pct < 0.1 else f"{pct:.1f}"
    text = f"{n:,} ({pct_str}\\%)"
    if marker_key and marker_key in MARKER_DEFS:
        return r"\textbf{" + text + "} " + MARKER_DEFS[marker_key]
    return text

# ------------------------------------------------------------------------
# Emit LaTeX.

PREAMBLE = r"""% --- Add to your document preamble (once) ---
% \usepackage{xcolor}
% \usepackage{fontawesome5}
% \usepackage{array}
% \usepackage{booktabs}
\definecolor{PolicyRed}{HTML}{A93226}
\definecolor{GapBlue}{HTML}{34495E}
\newcommand{\markWeak}{\textcolor{PolicyRed}{\faIcon{arrow-circle-down}}}
\newcommand{\markGap}{\textcolor{GapBlue}{\faIcon{minus-circle}}}
\newcolumntype{R}[1]{>{\raggedleft\arraybackslash}p{#1}}"""

n_schemes = len(SCHEME_ORDER)

lines = []
lines.append(r"\begin{table*}[t]")
lines.append(r"  \centering")
lines.append(
    r"  \caption{Apps loading HTTP resources by type and initiator scheme "
    r"relative to the total amount of dynamically analyzed apps that "
    r"include WebViews "
    r"(\markWeak~resource loads that result from systematic MC policy "
    r"weakening; "
    r"\markGap~resource loads that originate from enforcement gaps "
    r"compared to Chrome)."
    r"}"
)
lines.append(r"  \label{tab:resource-breakdown-apps}")
lines.append(r"  \footnotesize")
lines.append(
    r"  \begin{tabular}{p{2.1cm} | "
    + " ".join(["R{2.1cm}"] * n_schemes)
    + " | R{2.1cm}}"
)
lines.append(r"    \toprule")
lines.append(
    r"    & \multicolumn{" + str(n_schemes)
    + r"}{c}{\textbf{Initiator scheme}} & \\"
)
header = [r"\textbf{Loaded Resource}"] \
    + [r"\textbf{" + SCHEME_LABELS[s] + "}" for s in SCHEME_ORDER] \
    + [r"\textbf{Total}"]
lines.append("    " + " & ".join(header) + r" \\")
lines.append(r"    \midrule")

for item in ROW_LAYOUT:
    if item == "<midrule>":
        lines.append(r"    \midrule")
        continue
    if item == "<hline>":
        lines.append(r"    \hline")
        continue

    if isinstance(item, dict) and item.get("header_only"):
        cells = [item["label"]] + [""] * (n_schemes + 1)
        lines.append("    " + " & ".join(cells) + r" \\")
        continue

    label = item if isinstance(item, str) else item["label"]
    cells = [label]
    for s in SCHEME_ORDER:
        cells.append(fmt_cell(cell_count(item, s), lookup_marker(item, s)))
    cells.append(fmt_cell(row_total_count(item)))
    lines.append("    " + " & ".join(cells) + r" \\")

lines.append(r"    \midrule")
totals_row = [r"\textbf{Total}"]
for s in SCHEME_ORDER:
    n = int(apps_by_scheme.get(s, 0))
    totals_row.append(r"\textbf{" + f"{n:,}" + "}")
totals_row.append(r"\textbf{" + f"{int(df['package_name'].nunique()):,}" + "}")
lines.append("    " + " & ".join(totals_row) + r" \\")

lines.append(r"    \bottomrule")
lines.append(r"  \end{tabular}")
lines.append(r"\end{table*}")

latex = "\n".join(lines)
print(f"TOTAL_APPS = {TOTAL_APPS}")
print()
print(PREAMBLE)
print()
print(latex)


TOTAL_APPS = 34200

% --- Add to your document preamble (once) ---
% \usepackage{xcolor}
% \usepackage{fontawesome5}
% \usepackage{array}
% \usepackage{booktabs}
\definecolor{PolicyRed}{HTML}{A93226}
\definecolor{GapBlue}{HTML}{34495E}
\newcommand{\markWeak}{\textcolor{PolicyRed}{\faIcon{arrow-circle-down}}}
\newcommand{\markGap}{\textcolor{GapBlue}{\faIcon{minus-circle}}}
\newcolumntype{R}[1]{>{\raggedleft\arraybackslash}p{#1}}

\begin{table*}[t]
  \centering
  \caption{Apps loading HTTP resources by type and initiator scheme relative to the total amount of dynamically analyzed apps that include WebViews (\markWeak~resource loads that result from systematic MC policy weakening; \markGap~resource loads that originate from enforcement gaps compared to Chrome).}
  \label{tab:resource-breakdown-apps}
  \footnotesize
  \begin{tabular}{p{2.1cm} | R{2.1cm} R{2.1cm} R{2.1cm} R{2.1cm} R{2.1cm} | R{2.1cm}}
    \toprule
    & \multicolumn{5}{c}{\textbf{Initiator scheme}} & \\
    \textbf{Loaded 